## Check if the GPU is ON and viisble to PyTorch

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(f"  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
else:
    print("No CUDA GPU detected.")

## verify the input datasets' paths and all

In [ ]:
from pathlib import Path

INTENT_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/processed-intent-data"
)

SUMMARY_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/email-thread-summarization-dataset"
)

EMAIL_INTEL_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data"
)

print("========== DATASET PATH CHECK ==========\n")

paths_to_check = {
    "Intent train": INTENT_DIR / "train" / "intent_train.csv",
    "Intent validation": INTENT_DIR / "val" / "intent_val.csv",
    "Intent test": INTENT_DIR / "test" / "intent_test.csv",
    "Intent labeled": INTENT_DIR / "intent_labeled.csv",

    "Summary train": SUMMARY_DIR / "train" / "email_thread_summary_train.csv",
    "Summary validation": SUMMARY_DIR / "val" / "email_thread_summary_val.csv",
    "Summary test": SUMMARY_DIR / "test" / "email_thread_summary_test.csv",
    "Summary master": SUMMARY_DIR / "email_thread_summary_all.csv",

    "Email intelligence": EMAIL_INTEL_DIR,
}

for name, path in paths_to_check.items():
    print(f"{name:22} : {path.exists()}")

print("\n========== DATASET DIRECTORIES ==========")

print("\nIntent:")
for p in sorted(INTENT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(INTENT_DIR))

print("\nSummarization:")
for p in sorted(SUMMARY_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(SUMMARY_DIR))

## verify the columns and structure of the datasets before starting training

In [ ]:
import pandas as pd

INTENT_TRAIN_PATH = INTENT_DIR / "train" / "intent_train.csv"
INTENT_VAL_PATH = INTENT_DIR / "val" / "intent_val.csv"
INTENT_TEST_PATH = INTENT_DIR / "test" / "intent_test.csv"

intent_train_df = pd.read_csv(INTENT_TRAIN_PATH)
intent_val_df = pd.read_csv(INTENT_VAL_PATH)
intent_test_df = pd.read_csv(INTENT_TEST_PATH)

print("========== INTENT DATASET ==========")

print("\nTrain:")
print("Shape:", intent_train_df.shape)
print("Columns:", intent_train_df.columns.tolist())

print("\nValidation:")
print("Shape:", intent_val_df.shape)
print("Columns:", intent_val_df.columns.tolist())

print("\nTest:")
print("Shape:", intent_test_df.shape)
print("Columns:", intent_test_df.columns.tolist())

print("\n========== TRAIN SAMPLE ==========")
display(intent_train_df.head())

print("\n========== LABEL DISTRIBUTION ==========")

if "intent_label" in intent_train_df.columns:
    print(intent_train_df["intent_label"].value_counts().sort_index())
elif "label" in intent_train_df.columns:
    print(intent_train_df["label"].value_counts().sort_index())
else:
    print("No obvious intent label column found.")

## Establish the exact text-label mapping

In [ ]:
print("========== INTENT TRAINING COLUMNS ==========")
print(intent_train_df.columns.tolist())

print("\n========== DATA TYPES ==========")
print(intent_train_df.dtypes)

print("\n========== SAMPLE ==========")
display(
    intent_train_df[
        ["cleaned_body", "intent_label"]
    ].head(5)
)

print("\n========== LABEL NAMES ==========")

if "intent_name" in intent_train_df.columns:
    print(
        intent_train_df[
            ["intent_label", "intent_name"]
        ]
        .drop_duplicates()
        .sort_values("intent_label")
        .to_string(index=False)
    )
else:
    print("intent_name column is not present.")

# Intent classification model building

### Load the model and configure it for 6-class classification

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

LABEL_NAMES = {
    0: "REQUEST",
    1: "FOLLOW_UP",
    2: "INFORMATION",
    3: "ACKNOWLEDGEMENT",
    4: "COMPLAINT",
    5: "INVITATION",
}

id2label = {
    label_id: label_name
    for label_id, label_name in LABEL_NAMES.items()
}

label2id = {
    label_name: label_id
    for label_id, label_name in LABEL_NAMES.items()
}

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
intent_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

intent_model = intent_model.to(device)

print("\n========== MODEL SETUP ==========")
print("Model:", MODEL_NAME)
print("Number of labels:", len(LABEL_NAMES))
print("Device:", device)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("Primary GPU:", torch.cuda.get_device_name(0))

print("\n========== LABEL MAPPING ==========")
for label_id in sorted(id2label):
    print(f"{label_id}: {id2label[label_id]}")

print("\nModel loaded successfully.")

### Tokenizing the datasets

In [ ]:
from datasets import Dataset

MAX_LENGTH = 256

intent_train = Dataset.from_pandas(
    intent_train_df[["cleaned_body", "intent_label"]],
    preserve_index=False
)

intent_val = Dataset.from_pandas(
    intent_val_df[["cleaned_body", "intent_label"]],
    preserve_index=False
)

intent_test = Dataset.from_pandas(
    intent_test_df[["cleaned_body", "intent_label"]],
    preserve_index=False
)

def tokenize_intent(batch):
    return tokenizer(
        batch["cleaned_body"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

intent_train = intent_train.map(
    tokenize_intent,
    batched=True,
    desc="Tokenizing training data"
)

intent_val = intent_val.map(
    tokenize_intent,
    batched=True,
    desc="Tokenizing validation data"
)

intent_test = intent_test.map(
    tokenize_intent,
    batched=True,
    desc="Tokenizing test data"
)

intent_train = intent_train.rename_column(
    "intent_label",
    "labels"
)

intent_val = intent_val.rename_column(
    "intent_label",
    "labels"
)

intent_test = intent_test.rename_column(
    "intent_label",
    "labels"
)

print("========== TOKENIZED DATASETS ==========")
print("Train:", len(intent_train))
print("Validation:", len(intent_val))
print("Test:", len(intent_test))

print("\n========== SAMPLE ==========")
print(intent_train[0])

### Preparation of the data collator and evaluation metrics and finally configure the trainer and also calculate the class weights
Data collator - Divides the tokenized data into batches that can be fed to the model
Here we are using the data collator with padding i.e. different batches will have different sizes and hence we add additional PAD tokens to equalize the number of tokens in every batch

In [ ]:
# ============================================================
# DATA COLLATOR + CLASS WEIGHTS
# ============================================================

import torch
from transformers import DataCollatorWithPadding

# ------------------------------------------------------------
# Data collator
# ------------------------------------------------------------

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

print("========== DATA COLLATOR ==========")
print("Dynamic padding: Enabled")
print("Data collator:", type(data_collator).__name__)


# ------------------------------------------------------------
# Calculate class weights
# ------------------------------------------------------------

label_counts = (
    intent_train_df["intent_label"]
    .value_counts()
    .sort_index()
)

num_classes = len(LABEL_NAMES)
total_samples = len(intent_train_df)

class_weights = total_samples / (
    num_classes * label_counts
)

class_weights = class_weights.reindex(
    range(num_classes)
).fillna(0)

class_weights_tensor = torch.tensor(
    class_weights.values,
    dtype=torch.float32
)

print("\n========== CLASS DISTRIBUTION ==========")

for label_id in range(num_classes):
    print(
        f"{label_id}: "
        f"{LABEL_NAMES[label_id]:<16} "
        f"{label_counts[label_id]:>6,} samples"
    )

print("\n========== CLASS WEIGHTS ==========")

for label_id in range(num_classes):
    print(
        f"{label_id}: "
        f"{LABEL_NAMES[label_id]:<16} "
        f"{class_weights_tensor[label_id].item():.4f}"
    )

print("\n========== CLASS WEIGHT TENSOR ==========")
print(class_weights_tensor)

## What is Cross-Entropy Loss
This is what we are going to use when the model is being trained.
Lets say the mail that we have is : "Please send me the updated report.", then the actual class is REQUEST(0) but the model may predict different values with probabilites like for example : 
REQUEST          → 0.70
FOLLOW_UP        → 0.10
INFORMATION      → 0.08
ACKNOWLEDGEMENT  → 0.07
COMPLAINT        → 0.03
INVITATION       → 0.02

Here we can see the model is condierably confident that the intent is REQUEST so the loss is relatively lesser.
However if the probabilites are : 
REQUEST          → 0.05
FOLLOW_UP        → 0.10
INFORMATION      → 0.10
ACKNOWLEDGEMENT  → 0.65
COMPLAINT        → 0.05
INVITATION       → 0.05

Then we can see that the loss is very large.
Therefore Cross Entropy loss measures how wrong, the model's predicted probabilities are as compared to the actual class. 
During the training phase, the model tries to minimize this loss

Specifically here, we are using weighted cross-entropy loss function : This means that mistakes in predicting important/rare classes (which are the classes with higher class weights) result in a much higher penalty.

## Implement evaluation metrics, wegihted cross-entropy loss, training config and GPU config

In [ ]:
# ============================================================
# METRICS + WEIGHTED CROSS-ENTROPY LOSS
# ============================================================

import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)


# ------------------------------------------------------------
# Evaluation metrics
# ------------------------------------------------------------

def compute_intent_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision_macro, recall_macro, f1_macro, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    precision_weighted, recall_weighted, f1_weighted, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted,
    }


# ------------------------------------------------------------
# Weighted cross-entropy loss
# ------------------------------------------------------------

class WeightedTrainerLoss(nn.Module):

    def __init__(self, class_weights):
        super().__init__()

        self.register_buffer(
            "class_weights",
            class_weights
        )

        self.loss_function = nn.CrossEntropyLoss(
            weight=self.class_weights
        )

    def forward(self, logits, labels):

        return self.loss_function(
            logits,
            labels
        )


weighted_loss = WeightedTrainerLoss(
    class_weights_tensor
)


# ------------------------------------------------------------
# Verify configuration
# ------------------------------------------------------------

print("========== EVALUATION METRICS ==========")

print("Accuracy")
print("Macro Precision")
print("Macro Recall")
print("Macro F1")
print("Weighted Precision")
print("Weighted Recall")
print("Weighted F1")

print("\n========== WEIGHTED LOSS ==========")
print("Loss function:", type(weighted_loss.loss_function).__name__)
print("Number of classes:", len(class_weights_tensor))

print("\nClass weights:")
for label_id in range(len(LABEL_NAMES)):
    print(
        f"{label_id}: "
        f"{LABEL_NAMES[label_id]:<16} "
        f"{class_weights_tensor[label_id].item():.4f}"
    )

print("\nWeighted loss configured successfully.")

## Configure the trainer

In [ ]:
# ============================================================
# INTENT CLASSIFICATION — TRAINER CONFIGURATION
# ============================================================

import os
import torch
from transformers import Trainer, TrainingArguments


# ------------------------------------------------------------
# Custom Trainer with weighted cross-entropy
# ------------------------------------------------------------

class WeightedLossTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss = torch.nn.functional.cross_entropy(
            logits,
            labels,
            weight=class_weights_tensor.to(logits.device)
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

INTENT_OUTPUT_DIR = "/kaggle/working/intent_classification_model"

os.makedirs(
    INTENT_OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Training arguments
# ------------------------------------------------------------

training_args = TrainingArguments(

    output_dir=INTENT_OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    learning_rate=2e-5,

    # Batch configuration
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,

    # Evaluation
    eval_strategy="epoch",

    # Checkpointing
    save_strategy="epoch",
    save_total_limit=2,

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),

    # Logging
    logging_strategy="steps",
    logging_steps=100,

    # Reproducibility
    seed=42,

    # DataLoader
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # Reporting
    report_to="none"
)


# ------------------------------------------------------------
# Create Trainer
# ------------------------------------------------------------

intent_trainer = WeightedLossTrainer(

    model=intent_model,

    args=training_args,

    train_dataset=intent_train,

    eval_dataset=intent_val,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_intent_metrics,
)


# ------------------------------------------------------------
# Configuration verification
# ------------------------------------------------------------

print("========== TRAINER CONFIGURATION ==========")

print("Model:", MODEL_NAME)
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)

print(
    "Train batch size / GPU:",
    training_args.per_device_train_batch_size
)

print(
    "Effective train batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print(
    "Evaluation batch size / GPU:",
    training_args.per_device_eval_batch_size
)

print("FP16:", training_args.fp16)
print("Evaluation strategy:", training_args.eval_strategy)
print("Save strategy:", training_args.save_strategy)
print("Best-model metric:", training_args.metric_for_best_model)
print("Workers:", training_args.dataloader_num_workers)

print("\n========== DATASET SIZES ==========")
print("Train:", len(intent_train))
print("Validation:", len(intent_val))
print("Test:", len(intent_test))

print("\n========== GPU ==========")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

print("\nWeighted Trainer created successfully.")

## Checking on how to use both GPUs together

In [ ]:
# ============================================================
# MULTI-GPU ENVIRONMENT VERIFICATION
# ============================================================

import os
import torch

print("========== CUDA ENVIRONMENT ==========")

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA version:", torch.version.cuda)

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)

    print(f"\nGPU {i}")
    print("Name:", torch.cuda.get_device_name(i))
    print(
        "VRAM:",
        round(props.total_memory / (1024 ** 3), 2),
        "GB"
    )


print("\n========== DISTRIBUTED ENVIRONMENT ==========")

print(
    "LOCAL_RANK:",
    os.environ.get("LOCAL_RANK", "Not set")
)

print(
    "RANK:",
    os.environ.get("RANK", "Not set")
)

print(
    "WORLD_SIZE:",
    os.environ.get("WORLD_SIZE", "Not set")
)

print(
    "MASTER_ADDR:",
    os.environ.get("MASTER_ADDR", "Not set")
)

print(
    "MASTER_PORT:",
    os.environ.get("MASTER_PORT", "Not set")
)


print("\n========== PYTORCH DISTRIBUTED ==========")

print(
    "Distributed available:",
    torch.distributed.is_available()
)

print(
    "Distributed initialized:",
    (
        torch.distributed.is_initialized()
        if torch.distributed.is_available()
        else False
    )
)


print("\n========== CURRENT DEVICE ==========")

print("Current CUDA device:", torch.cuda.current_device())
print(
    "Current device name:",
    torch.cuda.get_device_name(torch.cuda.current_device())
)

## How will both GPUs be used
in order to make use of both GPUs, we will copy the training setpu into a python script and tehn launch that script with 2 processes for each of the GPUs. Every GPU will handle its own batch of the training batch 

## Check if accelerate is installed and available

In [ ]:
import accelerate
import transformers

print("========== TRAINING LIBRARY VERSIONS ==========")
print("Accelerate version :", accelerate.__version__)
print("Transformers version:", transformers.__version__)

print("\nAccelerate is available: True")

## Creating a .py script from the entire tainer config

### First create the .py file in the working folder

In [ ]:
%%writefile /kaggle/working/train_intent_distributed.py

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from accelerate import Accelerator
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

MODEL_NAME = "distilbert-base-uncased"

# Permanent Kaggle input dataset
BASE_DATA_PATH = (
    "/kaggle/input/datasets/prishabhkumar/"
    "processed-intent-data"
)

TRAIN_PATH = (
    f"{BASE_DATA_PATH}/train/intent_train.csv"
)

VAL_PATH = (
    f"{BASE_DATA_PATH}/val/intent_val.csv"
)

TEST_PATH = (
    f"{BASE_DATA_PATH}/test/intent_test.csv"
)

# Training configuration
NUM_EPOCHS = 3

# IMPORTANT:
# This is the batch size PER GPU.
BATCH_SIZE_PER_GPU = 16

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

MAX_LENGTH = 128

NUM_WORKERS = 2

# Output directory
OUTPUT_DIR = (
    "/kaggle/working/distilbert_intent_distributed"
)

# Actual columns in our dataset
TEXT_COLUMN = "cleaned_body"
LABEL_COLUMN = "intent_label"


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# 3. INITIALIZE ACCELERATE
# ============================================================

accelerator = Accelerator(
    mixed_precision="fp16"
)

device = accelerator.device


if accelerator.is_main_process:

    print("=" * 75)
    print("DISTRIBUTED TRAINING INITIALIZATION")
    print("=" * 75)

    print(f"Device                  : {device}")
    print(
        f"Number of processes     : "
        f"{accelerator.num_processes}"
    )

    print(
        f"Mixed precision        : "
        f"{accelerator.mixed_precision}"
    )

    print(
        f"Batch size per GPU     : "
        f"{BATCH_SIZE_PER_GPU}"
    )

    print(
        f"Effective batch size   : "
        f"{BATCH_SIZE_PER_GPU * accelerator.num_processes}"
    )

    print(
        f"Number of epochs       : "
        f"{NUM_EPOCHS}"
    )

    print(
        f"Learning rate          : "
        f"{LEARNING_RATE}"
    )

    print(
        f"Model                  : "
        f"{MODEL_NAME}"
    )

    print("=" * 75)


# ============================================================
# 4. VERIFY GPU CONFIGURATION
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("GPU CONFIGURATION")
    print("=" * 75)

    print(
        f"CUDA available : "
        f"{torch.cuda.is_available()}"
    )

    print(
        f"CUDA devices   : "
        f"{torch.cuda.device_count()}"
    )

    for gpu_id in range(torch.cuda.device_count()):

        print(
            f"GPU {gpu_id}: "
            f"{torch.cuda.get_device_name(gpu_id)}"
        )

    print("=" * 75)


# ============================================================
# 5. LOAD DATASETS
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("LOADING DATASETS")
    print("=" * 75)

    print(f"Train: {TRAIN_PATH}")
    print(f"Val  : {VAL_PATH}")
    print(f"Test : {TEST_PATH}")


train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)


# ============================================================
# 6. CLEAN DATA
# ============================================================

def clean_dataframe(df):

    df = df.copy()

    df = df.dropna(
        subset=[
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    )

    df[TEXT_COLUMN] = (
        df[TEXT_COLUMN]
        .astype(str)
    )

    df[LABEL_COLUMN] = (
        pd.to_numeric(
            df[LABEL_COLUMN],
            errors="coerce"
        )
    )

    df = df.dropna(
        subset=[LABEL_COLUMN]
    )

    df[LABEL_COLUMN] = (
        df[LABEL_COLUMN]
        .astype(int)
    )

    return df


train_df = clean_dataframe(train_df)
val_df = clean_dataframe(val_df)
test_df = clean_dataframe(test_df)


if accelerator.is_main_process:

    print("\nDataset sizes after cleaning:")

    print(
        f"Training   : {len(train_df)}"
    )

    print(
        f"Validation : {len(val_df)}"
    )

    print(
        f"Test       : {len(test_df)}"
    )


# ============================================================
# 7. DETERMINE LABEL INFORMATION
# ============================================================

all_labels = sorted(
    set(train_df[LABEL_COLUMN].unique())
    |
    set(val_df[LABEL_COLUMN].unique())
    |
    set(test_df[LABEL_COLUMN].unique())
)

num_labels = len(all_labels)

# We preserve the existing integer label IDs.
#
# Example:
# 0 -> REQUEST
# 1 -> ...
# 2 -> INFORMATION
# etc.

label_id_to_name = {}

for label_id in all_labels:

    matching_rows = train_df[
        train_df[LABEL_COLUMN] == label_id
    ]

    if len(matching_rows) == 0:

        matching_rows = val_df[
            val_df[LABEL_COLUMN] == label_id
        ]

    if len(matching_rows) == 0:

        matching_rows = test_df[
            test_df[LABEL_COLUMN] == label_id
        ]

    if "intent_name" in matching_rows.columns:

        label_name = (
            matching_rows["intent_name"]
            .iloc[0]
        )

    else:

        label_name = str(label_id)

    label_id_to_name[int(label_id)] = str(
        label_name
    )


id2label = {
    int(label_id): label_name
    for label_id, label_name
    in label_id_to_name.items()
}

label2id = {
    label_name: int(label_id)
    for label_id, label_name
    in id2label.items()
}


if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("LABEL INFORMATION")
    print("=" * 75)

    print(
        f"Number of classes : "
        f"{num_labels}"
    )

    print("\nLabel mapping:")

    for label_id in all_labels:

        print(
            f"{label_id:3d} -> "
            f"{id2label[label_id]}"
        )

    print("=" * 75)


# ============================================================
# 8. CONVERT DATAFRAMES TO HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[
        [
            TEXT_COLUMN,
            LABEL_COLUMN
        ]
    ],
    preserve_index=False
)


# ============================================================
# 9. LOAD TOKENIZER
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("LOADING TOKENIZER")
    print("=" * 75)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 10. TOKENIZATION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH
    )


if accelerator.is_main_process:

    print("Tokenizing training data...")


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing training"
)


if accelerator.is_main_process:

    print("Tokenizing validation data...")


val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation"
)


if accelerator.is_main_process:

    print("Tokenizing test data...")


test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing test"
)


# ============================================================
# 11. PREPARE DATASET COLUMNS
# ============================================================

train_dataset = train_dataset.remove_columns(
    [TEXT_COLUMN]
)

val_dataset = val_dataset.remove_columns(
    [TEXT_COLUMN]
)

test_dataset = test_dataset.remove_columns(
    [TEXT_COLUMN]
)


train_dataset = train_dataset.rename_column(
    LABEL_COLUMN,
    "labels"
)

val_dataset = val_dataset.rename_column(
    LABEL_COLUMN,
    "labels"
)

test_dataset = test_dataset.rename_column(
    LABEL_COLUMN,
    "labels"
)


train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")


# ============================================================
# 12. DATA COLLATOR
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# 13. DATALOADERS
# ============================================================

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=BATCH_SIZE_PER_GPU,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=BATCH_SIZE_PER_GPU,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=BATCH_SIZE_PER_GPU,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# ============================================================
# 14. CALCULATE CLASS WEIGHTS
# ============================================================

train_labels = (
    train_df[LABEL_COLUMN]
    .values
)

class_counts = np.bincount(
    train_labels,
    minlength=max(all_labels) + 1
)


# Standard balanced-class weighting:
#
# weight_i = N / (number_of_classes * count_i)

class_weights = (
    len(train_labels)
    /
    (
        num_labels
        *
        np.maximum(
            class_counts,
            1
        )
    )
)


class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)


if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("CLASS DISTRIBUTION AND WEIGHTS")
    print("=" * 75)

    for label_id in all_labels:

        print(
            f"Class {label_id:3d} | "
            f"Count: {class_counts[label_id]:6d} | "
            f"Weight: {class_weights[label_id]:.6f} | "
            f"Label: {id2label[label_id]}"
        )

    print("=" * 75)


# ============================================================
# 15. LOAD DISTILBERT
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("LOADING DISTILBERT")
    print("=" * 75)


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={
        int(k): str(v)
        for k, v in id2label.items()
    },
    label2id={
        str(k): int(v)
        for k, v in label2id.items()
    }
)


# ============================================================
# 16. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 17. PREPARE FOR DISTRIBUTED TRAINING
# ============================================================

(
    model,
    optimizer,
    train_dataloader,
    val_dataloader,
    test_dataloader
) = accelerator.prepare(
    model,
    optimizer,
    train_dataloader,
    val_dataloader,
    test_dataloader
)


# Class weights must be on the same device as the model.

class_weights = class_weights.to(device)


# ============================================================
# 18. WEIGHTED CROSS-ENTROPY LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 19. LEARNING RATE SCHEDULER
# ============================================================

num_update_steps_per_epoch = len(
    train_dataloader
)

max_train_steps = (
    NUM_EPOCHS
    *
    num_update_steps_per_epoch
)

num_warmup_steps = int(
    WARMUP_RATIO
    *
    max_train_steps
)


lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps
)


# ============================================================
# 20. METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    predictions,
    labels
):

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(
            weighted_precision
        ),
        "weighted_recall": float(
            weighted_recall
        ),
        "weighted_f1": float(
            weighted_f1
        )
    }


# ============================================================
# 21. EVALUATION FUNCTION
# ============================================================

def evaluate(
    model,
    dataloader
):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    all_predictions = []
    all_labels = []

    for batch in dataloader:

        with torch.no_grad():

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            logits = outputs.logits

            loss = criterion(
                logits,
                batch["labels"]
            )

        total_loss += loss.item()
        total_batches += 1

        predictions = torch.argmax(
            logits,
            dim=-1
        )

        # Gather predictions/labels from all GPUs.

        predictions = accelerator.gather_for_metrics(
            predictions
        )

        labels = accelerator.gather_for_metrics(
            batch["labels"]
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_labels.append(
            labels.cpu()
        )

    all_predictions = torch.cat(
        all_predictions
    ).numpy()

    all_labels = torch.cat(
        all_labels
    ).numpy()

    metrics = calculate_metrics(
        all_predictions,
        all_labels
    )

    metrics["loss"] = (
        total_loss
        /
        max(total_batches, 1)
    )

    return (
        metrics,
        all_predictions,
        all_labels
    )


# ============================================================
# 22. TRAINING
# ============================================================

best_val_macro_f1 = -float("inf")
best_epoch = 0

training_history = []


if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("STARTING DISTRIBUTED TRAINING")
    print("=" * 75)

    print(
        f"Processes / GPUs : "
        f"{accelerator.num_processes}"
    )

    print(
        f"Batch per GPU    : "
        f"{BATCH_SIZE_PER_GPU}"
    )

    print(
        f"Effective batch  : "
        f"{BATCH_SIZE_PER_GPU * accelerator.num_processes}"
    )

    print(
        f"Epochs           : "
        f"{NUM_EPOCHS}"
    )

    print("=" * 75)


for epoch in range(NUM_EPOCHS):

    model.train()

    total_train_loss = 0.0

    for step, batch in enumerate(
        train_dataloader
    ):

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs.logits

        loss = criterion(
            logits,
            batch["labels"]
        )

        accelerator.backward(loss)

        clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        lr_scheduler.step()

        total_train_loss += loss.item()

        if (
            accelerator.is_main_process
            and
            (
                step == 0
                or
                (step + 1) % 100 == 0
                or
                step == len(train_dataloader) - 1
            )
        ):

            print(
                f"Epoch "
                f"{epoch + 1}/{NUM_EPOCHS} | "
                f"Step "
                f"{step + 1}/{len(train_dataloader)} | "
                f"Loss: {loss.item():.4f}"
            )

    # --------------------------------------------------------
    # Training loss
    # --------------------------------------------------------

    train_loss_tensor = torch.tensor(
        total_train_loss,
        device=device
    )

    train_loss_tensor = accelerator.reduce(
        train_loss_tensor,
        reduction="sum"
    )

    train_loss = (
        train_loss_tensor.item()
        /
        (
            accelerator.num_processes
            *
            len(train_dataloader)
        )
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    (
        val_metrics,
        _,
        _
    ) = evaluate(
        model,
        val_dataloader
    )


    if accelerator.is_main_process:

        print("\n" + "-" * 75)

        print(
            f"EPOCH {epoch + 1} RESULTS"
        )

        print("-" * 75)

        print(
            f"Train Loss           : "
            f"{train_loss:.4f}"
        )

        print(
            f"Validation Loss      : "
            f"{val_metrics['loss']:.4f}"
        )

        print(
            f"Validation Accuracy  : "
            f"{val_metrics['accuracy']:.4f}"
        )

        print(
            f"Validation Macro F1  : "
            f"{val_metrics['macro_f1']:.4f}"
        )

        print(
            f"Validation Weighted F1: "
            f"{val_metrics['weighted_f1']:.4f}"
        )

        print(
            f"Learning Rate        : "
            f"{optimizer.param_groups[0]['lr']:.8f}"
        )

        print("-" * 75)


    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if (
        val_metrics["macro_f1"]
        >
        best_val_macro_f1
    ):

        best_val_macro_f1 = (
            val_metrics["macro_f1"]
        )

        best_epoch = epoch + 1

        accelerator.wait_for_everyone()

        unwrapped_model = (
            accelerator.unwrap_model(model)
        )

        if accelerator.is_main_process:

            best_model_dir = os.path.join(
                OUTPUT_DIR,
                "best_model"
            )

            os.makedirs(
                best_model_dir,
                exist_ok=True
            )

            unwrapped_model.save_pretrained(
                best_model_dir,
                save_function=accelerator.save
            )

            tokenizer.save_pretrained(
                best_model_dir
            )

            print(
                f"\n✓ Best model saved"
            )

            print(
                f"  Epoch     : {best_epoch}"
            )

            print(
                f"  Macro F1  : "
                f"{best_val_macro_f1:.4f}"
            )


    training_history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": (
                val_metrics["macro_precision"]
            ),
            "val_macro_recall": (
                val_metrics["macro_recall"]
            ),
            "val_macro_f1": (
                val_metrics["macro_f1"]
            ),
            "val_weighted_f1": (
                val_metrics["weighted_f1"]
            )
        }
    )


# ============================================================
# 23. SYNCHRONIZE PROCESSES
# ============================================================

accelerator.wait_for_everyone()


# ============================================================
# 24. LOAD BEST MODEL
# ============================================================

best_model_path = os.path.join(
    OUTPUT_DIR,
    "best_model"
)


if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("LOADING BEST MODEL FOR TEST EVALUATION")
    print("=" * 75)

    print(
        f"Best epoch : "
        f"{best_epoch}"
    )

    print(
        f"Best Val Macro F1 : "
        f"{best_val_macro_f1:.4f}"
    )


best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        best_model_path
    )
)


best_model = best_model.to(
    device
)


best_model, test_dataloader = (
    accelerator.prepare(
        best_model,
        test_dataloader
    )
)


# ============================================================
# 25. FINAL TEST EVALUATION
# ============================================================

(
    test_metrics,
    test_predictions,
    test_labels
) = evaluate(
    best_model,
    test_dataloader
)


# ============================================================
# 26. TEST RESULTS
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("FINAL TEST RESULTS")
    print("=" * 75)

    print(
        f"Test Loss             : "
        f"{test_metrics['loss']:.4f}"
    )

    print(
        f"Test Accuracy         : "
        f"{test_metrics['accuracy']:.4f}"
    )

    print(
        f"Test Macro Precision  : "
        f"{test_metrics['macro_precision']:.4f}"
    )

    print(
        f"Test Macro Recall     : "
        f"{test_metrics['macro_recall']:.4f}"
    )

    print(
        f"Test Macro F1         : "
        f"{test_metrics['macro_f1']:.4f}"
    )

    print(
        f"Test Weighted F1      : "
        f"{test_metrics['weighted_f1']:.4f}"
    )

    print("=" * 75)


# ============================================================
# 27. CLASSIFICATION REPORT
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("CLASSIFICATION REPORT")
    print("=" * 75)

    target_names = [
        id2label[label_id]
        for label_id in all_labels
    ]

    print(
        classification_report(
            test_labels,
            test_predictions,
            labels=all_labels,
            target_names=target_names,
            zero_division=0
        )
    )


# ============================================================
# 28. CONFUSION MATRIX
# ============================================================

if accelerator.is_main_process:

    print("\n" + "=" * 75)
    print("CONFUSION MATRIX")
    print("=" * 75)

    cm = confusion_matrix(
        test_labels,
        test_predictions,
        labels=all_labels
    )

    print(cm)


# ============================================================
# 29. SAVE RESULTS
# ============================================================

if accelerator.is_main_process:

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Label mapping
    # --------------------------------------------------------

    with open(
        os.path.join(
            OUTPUT_DIR,
            "label_mapping.json"
        ),
        "w"
    ) as f:

        json.dump(
            {
                "id2label": {
                    str(k): str(v)
                    for k, v in id2label.items()
                },
                "label2id": {
                    str(k): int(v)
                    for k, v in label2id.items()
                }
            },
            f,
            indent=4
        )


    # --------------------------------------------------------
    # Final results
    # --------------------------------------------------------

    final_results = {

        "model_name": MODEL_NAME,

        "best_epoch": best_epoch,

        "best_validation_macro_f1": (
            best_val_macro_f1
        ),

        "test_metrics": test_metrics,

        "num_labels": num_labels,

        "num_training_samples": len(train_df),

        "num_validation_samples": len(val_df),

        "num_test_samples": len(test_df),

        "epochs": NUM_EPOCHS,

        "batch_size_per_gpu": (
            BATCH_SIZE_PER_GPU
        ),

        "num_gpus": (
            accelerator.num_processes
        ),

        "effective_batch_size": (
            BATCH_SIZE_PER_GPU
            *
            accelerator.num_processes
        ),

        "learning_rate": LEARNING_RATE,

        "max_length": MAX_LENGTH,

        "mixed_precision": (
            accelerator.mixed_precision
        )
    }


    with open(
        os.path.join(
            OUTPUT_DIR,
            "final_results.json"
        ),
        "w"
    ) as f:

        json.dump(
            final_results,
            f,
            indent=4
        )


    # --------------------------------------------------------
    # Training history
    # --------------------------------------------------------

    history_df = pd.DataFrame(
        training_history
    )

    history_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "training_history.csv"
        ),
        index=False
    )


    # --------------------------------------------------------
    # FINAL SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 75)
    print("TRAINING COMPLETE")
    print("=" * 75)

    print(
        f"Best epoch              : "
        f"{best_epoch}"
    )

    print(
        f"Best validation Macro F1: "
        f"{best_val_macro_f1:.4f}"
    )

    print(
        f"Final test Macro F1     : "
        f"{test_metrics['macro_f1']:.4f}"
    )

    print(
        f"Final test accuracy     : "
        f"{test_metrics['accuracy']:.4f}"
    )

    print(
        f"\nBest model saved to:"
    )

    print(
        best_model_path
    )

    print(
        f"\nResults saved to:"
    )

    print(
        OUTPUT_DIR
    )

    print("=" * 75)


accelerator.wait_for_everyone()

### Verification

In [ ]:
import py_compile

script_path = "/kaggle/working/train_intent_distributed.py"

try:
    py_compile.compile(
        script_path,
        doraise=True
    )
    print("✓ Script syntax check passed")
    print(f"✓ Script exists: {script_path}")

except Exception as e:
    print("✗ Script syntax check failed")
    print(e)

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("Launching distributed training with 2 GPUs.")

# Launch the distributed training using both GPUs together

In [ ]:
!accelerate launch \
    --num_processes 2 \
    --mixed_precision fp16 \
    /kaggle/working/train_intent_distributed.py

# Final Model Performance

> **Best Model:** DistilBERT — Epoch 3  
> **Task:** 6-Class Email Intent Classification  
> **Training:** Distributed training on 2 × NVIDIA Tesla T4 GPUs

---

## Final Results

| Metric | Result |
|:---|---:|
| **Best Epoch** | **3** |
| **Best Validation Macro F1** | **0.8110** |
| **Final Test Macro F1** | **0.8091** |
| **Final Test Accuracy** | **87.20%** |

---

## Validation Macro F1 Progression

The model demonstrated **continuous improvement throughout training**, with the validation Macro F1 increasing consistently across all three epochs.

| Epoch | Validation Macro F1 | Improvement |
|:---:|---:|---:|
| **1** | 0.6715 | — |
| **2** | 0.7729 | +0.1014 |
| **3** | **0.8110** | +0.0381 |

### Performance Trend

```text
Epoch 1  ██████████████████████████████  0.6715
Epoch 2  ███████████████████████████████████  0.7729
Epoch 3  ███████████████████████████████████████  0.8110

## Saving the trained model

### Verify if the model exists at the mentioned path

In [ ]:
import os

model_dir = "/kaggle/working/distilbert_intent_distributed/best_model"

print("Model directory exists:", os.path.exists(model_dir))

if os.path.exists(model_dir):
    print("\nModel files:")
    for file in os.listdir(model_dir):
        print(" -", file)

### Create a .zip backup of the model folder

In [ ]:
import shutil

model_dir = "/kaggle/working/distilbert_intent_distributed/best_model"

zip_path = shutil.make_archive(
    "/kaggle/working/distilbert_intent_classifier",
    "zip",
    model_dir
)

print("Model archived at:")
print(zip_path)

### Create a small results file to store all information regarding the model's performance during training

In [ ]:
import json

results = {
    "model": "distilbert-base-uncased",
    "task": "email intent classification",
    "num_classes": 6,
    "classes": {
        "0": "REQUEST",
        "1": "FOLLOW_UP",
        "2": "INFORMATION",
        "3": "ACKNOWLEDGEMENT",
        "4": "COMPLAINT",
        "5": "INVITATION"
    },
    "epochs": 3,
    "batch_size_per_gpu": 16,
    "effective_batch_size": 32,
    "mixed_precision": "fp16",
    "num_gpus": 2,
    "best_validation_macro_f1": 0.8110,
    "test_accuracy": 0.8720,
    "test_macro_f1": 0.8091,
    "test_weighted_f1": 0.8750
}

with open(
    "/kaggle/working/intent_classifier_results.json",
    "w"
) as f:
    json.dump(results, f, indent=4)

print("Results saved.")

# NER Training Phase


## Check what data is available for NER

In [ ]:
import os

ner_base = "/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data"

print("=" * 70)
print("EMAIL INTELLIGENCE PROCESSED DATA")
print("=" * 70)

for root, dirs, files in os.walk(ner_base):
    level = root.replace(ner_base, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")